# CZSO IKZ – audit false negatives (v10 fixed)

Tato opravená verze je **samostatná** a nečeká natvrdo, že soubory leží přímo vedle notebooku.

Co dělá:
- automaticky dohledá `selected_relevant_catalogue.csv` nebo `.xlsx`,
- automaticky dohledá `ikz_a_to_z_manifest.csv` nebo `.xlsx`,
- volitelně načte `dataset_profiles_manifest.csv`,
- vytvoří auditní tabulky možných falešných negativů,
- uloží je do složky `audit_false_negatives`.

Pokud chceš použít vlastní cesty, stačí je ručně vyplnit v první code buňce.

In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import re
import pandas as pd
from IPython.display import display

# ====== Volitelné ruční přepsání cest ======
BASE_DIR = Path.cwd()
SHORTLIST_PATH = None              # např. BASE_DIR / 'selected_relevant_catalogue.csv'
FINAL_MANIFEST_PATH = None         # např. BASE_DIR / 'czso_ikz_a_to_z_output' / 'ikz_a_to_z_manifest.csv'
PROFILES_MANIFEST_PATH = None      # např. BASE_DIR.parent / 'czso_preview' / 'czso_dataset_profiles' / 'dataset_profiles_manifest.csv'
OUT_DIR = BASE_DIR / 'audit_false_negatives'
TARGET_YEARS = (2023, 2024)

YEAR_RE = re.compile(r"(?<!\d)(19\d{2}|20\d{2}|2100)(?!\d)")
KEY_ALIASES = {
    "obec_kod", "kod_obce", "kod_obce_zuj", "zuj_kod", "op_obec_kod", "doj_obec_kod",
    "uzemi_kod", "vuzemi_kod", "idobce", "id_obce", "uzemi", "vuzemi",
}


def _candidate_roots(base_dir: Path) -> list[Path]:
    roots = [
        base_dir,
        base_dir / 'czso_ikz_a_to_z_output',
        base_dir / 'czso_dataset_profiles',
        base_dir.parent,
        base_dir.parent / 'czso_ikz_a_to_z_output',
        base_dir.parent / 'czso_dataset_profiles',
    ]
    out = []
    seen = set()
    for r in roots:
        try:
            rr = r.resolve()
        except Exception:
            rr = r
        if rr not in seen:
            seen.add(rr)
            out.append(r)
    return out


def _find_file(base_dir: Path, filenames: list[str]) -> Path | None:
    roots = _candidate_roots(base_dir)

    # 1) Nejdřív běžná místa
    for root in roots:
        for name in filenames:
            p = root / name
            if p.exists():
                return p

    # 2) Pak omezený rekurzivní fallback v base_dir a parent
    search_roots = []
    for root in [base_dir, base_dir.parent]:
        if root.exists() and root not in search_roots:
            search_roots.append(root)

    for root in search_roots:
        for name in filenames:
            try:
                matches = list(root.rglob(name))
            except Exception:
                matches = []
            if matches:
                matches = sorted(matches, key=lambda p: (len(p.parts), str(p).lower()))
                return matches[0]

    return None


def _safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    encodings = ['utf-8-sig', 'utf-8', 'cp1250', 'windows-1250', 'cp1252', 'latin-1']
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err


def read_table_auto(path: Path, *, preferred_sheet: str | None = None) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return _safe_read_csv(path, dtype={"dataset_id": "string"})
    if suffix in {'.xlsx', '.xlsm', '.xls'}:
        if preferred_sheet is not None:
            try:
                return pd.read_excel(path, sheet_name=preferred_sheet, dtype={"dataset_id": "string"})
            except Exception:
                pass
        try:
            xl = pd.ExcelFile(path)
            sheet = preferred_sheet if preferred_sheet in xl.sheet_names else xl.sheet_names[0]
            return pd.read_excel(path, sheet_name=sheet, dtype={"dataset_id": "string"})
        except Exception:
            return pd.read_excel(path, dtype={"dataset_id": "string"})
    raise ValueError(f'Nepodporovaný formát: {path}')


def resolve_input_paths(
    *,
    base_dir: Path,
    shortlist_path: Path | None,
    final_manifest_path: Path | None,
    profiles_manifest_path: Path | None,
) -> dict[str, Path | None]:
    resolved = {}

    if shortlist_path is not None:
        resolved['shortlist'] = Path(shortlist_path)
    else:
        resolved['shortlist'] = _find_file(base_dir, ['selected_relevant_catalogue.csv', 'selected_relevant_catalogue.xlsx'])

    if final_manifest_path is not None:
        resolved['final_manifest'] = Path(final_manifest_path)
    else:
        resolved['final_manifest'] = _find_file(base_dir, ['ikz_a_to_z_manifest.csv', 'ikz_a_to_z_manifest.xlsx'])

    if profiles_manifest_path is not None:
        resolved['profiles_manifest'] = Path(profiles_manifest_path)
    else:
        resolved['profiles_manifest'] = _find_file(base_dir, ['dataset_profiles_manifest.csv'])

    return resolved


def parse_years_any(value: Any) -> set[int]:
    if value is None:
        return set()
    try:
        if pd.isna(value):
            return set()
    except Exception:
        pass
    years = set()
    for x in YEAR_RE.findall(str(value)):
        y = int(x)
        if 1990 <= y <= 2035:
            years.add(y)
    return years


def extract_years_from_profile_row(row: pd.Series) -> set[int]:
    years: set[int] = set()
    for col in ['years_sample', 'years_min', 'years_max', 'start_catalog', 'end_catalog', 'source_title', 'dataset_title', 'selected_file']:
        if col in row.index:
            years |= parse_years_any(row[col])
    return years


def read_head_columns(anchor_dir: Path, relpath: Any) -> set[str]:
    if relpath is None:
        return set()
    try:
        if pd.isna(relpath):
            return set()
    except Exception:
        pass
    p = anchor_dir / str(relpath)
    if not p.exists():
        return set()
    try:
        cols = _safe_read_csv(p, nrows=1).columns
        return {str(c).strip().lower().replace('"', '').replace('﻿', '') for c in cols}
    except Exception:
        return set()


def run_audit(
    *,
    base_dir: Path,
    shortlist_path: Path,
    final_manifest_path: Path,
    profiles_manifest_path: Path | None = None,
    target_years: tuple[int, int] = (2023, 2024),
) -> dict[str, pd.DataFrame]:
    shortlist_sheet = 'excluded' if shortlist_path.suffix.lower() in {'.xlsx', '.xlsm', '.xls'} else None
    final_sheet = 'manifest' if final_manifest_path.suffix.lower() in {'.xlsx', '.xlsm', '.xls'} else None

    shortlist = read_table_auto(shortlist_path, preferred_sheet=shortlist_sheet)
    final = read_table_auto(final_manifest_path, preferred_sheet=final_sheet)

    for df in (shortlist, final):
        if 'dataset_id' in df.columns:
            df['dataset_id'] = df['dataset_id'].astype('string').str.strip()

    final_ids = set(final['dataset_id'].dropna()) if 'dataset_id' in final.columns else set()
    outputs: dict[str, pd.DataFrame] = {}

    if profiles_manifest_path is None or not profiles_manifest_path.exists():
        missing = shortlist.loc[~shortlist['dataset_id'].isin(final_ids)].copy() if 'dataset_id' in shortlist.columns else shortlist.copy()
        outputs['missing_from_final_simple'] = missing
        return outputs

    prof = read_table_auto(profiles_manifest_path)
    if 'dataset_id' in prof.columns:
        prof['dataset_id'] = prof['dataset_id'].astype('string').str.strip()

    prof['profile_years'] = prof.apply(extract_years_from_profile_row, axis=1)
    prof['has_target_years_profile'] = prof['profile_years'].apply(lambda s: all(y in s for y in target_years))

    preview_col = 'preview_csv' if 'preview_csv' in prof.columns else None
    anchor_dir = profiles_manifest_path.parent
    if preview_col is not None:
        prof['head_columns'] = prof[preview_col].apply(lambda p: read_head_columns(anchor_dir, p))
    else:
        prof['head_columns'] = [set() for _ in range(len(prof))]

    prof['has_pk_from_head'] = prof['head_columns'].apply(lambda cols: bool(cols & KEY_ALIASES))
    prof['pk_from_head'] = prof['head_columns'].apply(lambda cols: ', '.join(sorted(cols & KEY_ALIASES)) if cols else None)
    prof['profile_ok'] = prof.get('status', pd.Series(index=prof.index, dtype=object)).astype(str).str.startswith('OK')

    keep_cols = [c for c in ['dataset_id', 'title', 'relevance_pro_IKZ', 'relevance_pro_ikz', 'uzemni_uroven', 'latest_year_hint', 'excluded_historical'] if c in shortlist.columns]
    shortlist_small = shortlist[keep_cols].drop_duplicates('dataset_id') if keep_cols else shortlist[['dataset_id']].drop_duplicates('dataset_id')
    audit = prof.merge(shortlist_small, on='dataset_id', how='left')
    audit['in_final_manifest'] = audit['dataset_id'].isin(final_ids)

    strong = audit[
        audit['profile_ok']
        & audit['has_target_years_profile']
        & audit['has_pk_from_head']
        & ~audit['in_final_manifest']
    ].copy()

    soft = audit[
        audit['profile_ok']
        & audit['has_target_years_profile']
        & ~audit['in_final_manifest']
    ].copy()

    technical = audit[
        ~audit['profile_ok'] & ~audit['in_final_manifest']
    ].copy()

    historical = audit[
        audit['profile_years'].apply(lambda s: len(s) > 0 and max(s) < 2020)
        & ~audit['in_final_manifest']
    ].copy()

    outputs['false_negative_strong'] = strong
    outputs['false_negative_soft'] = soft
    outputs['technical_failures'] = technical
    outputs['correct_historical_exclusions'] = historical
    return outputs


resolved_paths = resolve_input_paths(
    base_dir=BASE_DIR,
    shortlist_path=SHORTLIST_PATH,
    final_manifest_path=FINAL_MANIFEST_PATH,
    profiles_manifest_path=PROFILES_MANIFEST_PATH,
)

print('Base dir   :', BASE_DIR)
for key, value in resolved_paths.items():
    print(f'{key:13}:', value)

if resolved_paths['shortlist'] is None:
    raise FileNotFoundError(
        'Nepodařilo se dohledat selected_relevant_catalogue.csv ani .xlsx. '
        'Vyplň SHORTLIST_PATH ručně v první buňce.'
    )
if resolved_paths['final_manifest'] is None:
    raise FileNotFoundError(
        'Nepodařilo se dohledat ikz_a_to_z_manifest.csv ani .xlsx. '
        'Vyplň FINAL_MANIFEST_PATH ručně v první buňce.'
    )

Base dir   : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle
shortlist    : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\selected_relevant_catalogue.csv
final_manifest: C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\czso_ikz_a_to_z_output\ikz_a_to_z_manifest.csv
profiles_manifest: C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_preview\czso_dataset_profiles\dataset_profiles_manifest.csv


In [4]:
outputs = run_audit(
    base_dir=BASE_DIR,
    shortlist_path=resolved_paths['shortlist'],
    final_manifest_path=resolved_paths['final_manifest'],
    profiles_manifest_path=resolved_paths['profiles_manifest'],
    target_years=TARGET_YEARS,
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in outputs.items():
    out_path = OUT_DIR / f'{name}.csv'
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(name, len(df), '->', out_path)

list(outputs.keys())

false_negative_strong 0 -> C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\audit_false_negatives\false_negative_strong.csv
false_negative_soft 1 -> C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\audit_false_negatives\false_negative_soft.csv
technical_failures 12 -> C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\audit_false_negatives\technical_failures.csv
correct_historical_exclusions 11 -> C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_v10_display_fixed_clean_bundle\audit_false_negatives\correct_historical_exclusions.csv


['false_negative_strong',
 'false_negative_soft',
 'technical_failures',
 'correct_historical_exclusions']

In [5]:
summary_rows = []
for name, df in outputs.items():
    summary_rows.append({
        'table': name,
        'rows': len(df),
        'saved_to': str(OUT_DIR / f'{name}.csv'),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,table,rows,saved_to
0,false_negative_strong,0,C:\Users\jakub.sekanina\projekt_index_kvality_...
1,false_negative_soft,1,C:\Users\jakub.sekanina\projekt_index_kvality_...
2,technical_failures,12,C:\Users\jakub.sekanina\projekt_index_kvality_...
3,correct_historical_exclusions,11,C:\Users\jakub.sekanina\projekt_index_kvality_...


In [6]:
for name, df in outputs.items():
    print(f'### {name}:', len(df))
    display(df.head(20))

### false_negative_strong: 0


,order,priority,rodina_datasetu,dataset_id,dataset_title,source_kind,source_code,source_title,selected_file,selected_sheet,...,profile_years,has_target_years_profile,head_columns,has_pk_from_head,pk_from_head,profile_ok,title,relevance_pro_IKZ,uzemni_uroven_y,in_final_manifest


### false_negative_soft: 1


,order,priority,rodina_datasetu,dataset_id,dataset_title,source_kind,source_code,source_title,selected_file,selected_sheet,...,profile_years,has_target_years_profile,head_columns,has_pk_from_head,pk_from_head,profile_ok,title,relevance_pro_IKZ,uzemni_uroven_y,in_final_manifest
137,138,C,Registr ekonomických subjektů - seznam subjektů,res_data,Registr ekonomických subjektů - seznam subjektů,LKOD,resource#1,{'cs': 'Seznam ekonomických subjektů v CSV'},ds_res_data.csv,NaN,...,"{2021, 2022, 2023, 2024, 2025, 2026}",True,"{nace2025, cobce_text, ulice_text, ciss2010, c...",False,,True,Registr ekonomických subjektů - seznam subjektů,podpůrná / kontextová,nespecifikováno / mix,False


### technical_failures: 12


,order,priority,rodina_datasetu,dataset_id,dataset_title,source_kind,source_code,source_title,selected_file,selected_sheet,...,profile_years,has_target_years_profile,head_columns,has_pk_from_head,pk_from_head,profile_ok,title,relevance_pro_IKZ,uzemni_uroven_y,in_final_manifest
89,90,B,Ekonomické subjekty podle odvětví převažující ...,140133,Ekonomické subjekty podle odvětví převažující ...,DATASTAT,NaN,NaN,NaN,NaN,...,"{2025, 2010}",False,{},False,None,False,Ekonomické subjekty podle odvětví převažující ...,jádrová pro IKŽ,ORP,False
120,121,C,"Cizinci podle státního občanství, věku a pohlaví",290038r25,"Cizinci podle státního občanství, věku a pohla...",NaN,NaN,NaN,NaN,NaN,...,{2024},False,{},False,None,False,"Cizinci podle státního občanství, věku a pohla...",podpůrná / kontextová,okres,False
127,128,C,Indexy spotřebitelských cen,10022,Indexy spotřebitelských cen,DATASTAT,NaN,NaN,NaN,NaN,...,"{2025, 1995}",False,{},False,None,False,Indexy spotřebitelských cen,podpůrná / kontextová,ČR,False
128,129,C,"Indexy tržeb v odvětví dopravy, maloobchodu a ...",30030,"Indexy tržeb v odvětví dopravy, maloobchodu a ...",DATASTAT,NaN,NaN,NaN,NaN,...,"{2000, 2025}",False,{},False,None,False,"Indexy tržeb v odvětví dopravy, maloobchodu a ...",podpůrná / kontextová,ČR,False
129,130,C,Konjunkturální průzkumy,70013,Konjunkturální průzkumy,DATASTAT,NaN,NaN,NaN,NaN,...,"{2000, 2025}",False,{},False,None,False,Konjunkturální průzkumy,podpůrná / kontextová,ČR,False
132,133,C,Průměrné spotřebitelské ceny pohonných hmot - ...,phm_tydny,Průměrné spotřebitelské ceny pohonných hmot - ...,LKOD,resource#1,{'cs': 'Data v CSV'},NaN,NaN,...,"{2016, 2024}",False,{},False,None,False,Průměrné spotřebitelské ceny pohonných hmot - ...,podpůrná / kontextová,nespecifikováno / mix,False
133,134,C,Průměrné spotřebitelské ceny vybraných výrobků...,12052,Průměrné spotřebitelské ceny vybraných výrobků...,DATASTAT,NaN,NaN,NaN,NaN,...,"{2018, 2006}",False,{},False,None,False,Průměrné spotřebitelské ceny vybraných výrobků...,podpůrná / kontextová,kraj,False
135,136,C,Regionální účty za regiony soudržnosti a kraje,50101,Regionální účty za regiony soudržnosti a kraje,DATASTAT,NaN,NaN,NaN,NaN,...,"{2024, 1995}",False,{},False,None,False,Regionální účty za regiony soudržnosti a kraje,podpůrná / kontextová,kraj,False
144,145,C,Základní ukazatele národních účtů,50111,Základní ukazatele národních účtů,NaN,NaN,NaN,NaN,NaN,...,"{2024, 1990}",False,{},False,None,False,Základní ukazatele národních účtů,podpůrná / kontextová,nespecifikováno / mix,False
146,147,C,Vybavenost domácností informačními a komunikač...,60003,Vybavenost domácností informačními a komunikač...,DATASTAT,NaN,NaN,NaN,NaN,...,"{2025, 2007}",False,{},False,None,False,Vybavenost domácností informačními a komunikač...,podpůrná / kontextová,kraj,False


### correct_historical_exclusions: 11


,order,priority,rodina_datasetu,dataset_id,dataset_title,source_kind,source_code,source_title,selected_file,selected_sheet,...,profile_years,has_target_years_profile,head_columns,has_pk_from_head,pk_from_head,profile_ok,title,relevance_pro_IKZ,uzemni_uroven_y,in_final_manifest
1,2,A,Obyvatelstvo a domy v obcích podle výsledků sč...,170240-17,Obyvatelstvo a domy v obcích podle výsledků sč...,LKOD,resource#1,{'cs': 'Obyvatelstvo a domy v obcích podle výs...,OD_SLDB_HD_at.csv,NaN,...,"{2001, 2011, 1991}",False,"{okres_text, kraj, vuzemi_cis, kraj_text, stap...",True,vuzemi_kod,True,Obyvatelstvo a domy v obcích podle výsledků sč...,jádrová pro IKŽ,obec,False
2,3,A,Obyvatelstvo v základních sídelních jednotkách...,170241-17,Obyvatelstvo v základních sídelních jednotkách...,LKOD,resource#1,{'cs': 'Obyvatelstvo v základních sídelních je...,OD_SLD_ZSJD.csv,NaN,...,{2011},False,"{zsjd_kod, stapro_kod, hodnota, datum, obec_tx...",True,obec_kod,True,Obyvatelstvo v základních sídelních jednotkách...,jádrová pro IKŽ,podobecní úroveň (část obce/ZSJ),False
38,39,B,Výběr údajů ze SLDB 2011 za domy a byty,sldbdomy,Výběr údajů ze SLDB 2011 za domy a byty,LKOD,resource#1,{'cs': 'Domy a byty'},SLDB_DOMYBYTY.CSV,NaN,...,{2011},False,"{vse8172, vse7192, vse7183, vse71102, vse8114,...",False,,True,Výběr údajů ze SLDB 2011 za domy a byty,jádrová pro IKŽ,více úrovní (ČR–kraj–okres–obec),False
80,81,B,Výběr údajů ze SLDB 2011 za obyvatelstvo,SLDB-VYBER,Výběr údajů ze SLDB 2011 za obyvatelstvo,LKOD,resource#1,{'cs': 'Obyvatelstvo'},SLDB_OBYVATELSTVO.CSV,NaN,...,{2011},False,"{vse3171, vse4162, vse3142, vse4142, vse5163, ...",False,,True,Výběr údajů ze SLDB 2011 za obyvatelstvo,jádrová pro IKŽ,více úrovní (ČR–kraj–okres–obec),False
88,89,B,Výběr údajů ze SLDB 2011 za vyjížďku,sldbvyjizdka,Výběr údajů ze SLDB 2011 za vyjížďku,LKOD,resource#1,{'cs': 'Vyjížďka'},SLDB_VYJIZDKA.CSV,NaN,...,{2011},False,"{vse9151, vse9171, vse91101, vse9161, typuz_na...",False,,True,Výběr údajů ze SLDB 2011 za vyjížďku,jádrová pro IKŽ,více úrovní (ČR–kraj–okres–obec),False
103,104,B,"Základní výsledky Sčítání lidu, domů a bytů",sldb2011_zakladni,"Základní výsledky Sčítání lidu, domů a bytů 2011",LKOD,resource#1,{'cs': 'Základní výsledky - CSV soubor'},ds_sldb2011_zakladni.csv,NaN,...,{2011},False,"{u09, u08, u10, u02, u03, typuz_naz, uzcis, u0...",False,,True,"Základní výsledky Sčítání lidu, domů a bytů 2011",jádrová pro IKŽ,více úrovní (ČR–kraj–okres–obec),False
115,116,B,Výběr údajů ze SLDB 2011 za domácnosti,sldbdomac,Výběr údajů ze SLDB 2011 za domácnosti,LKOD,resource#1,{'cs': 'Domácnosti'},SLDB_DOMACNOSTI.CSV,NaN,...,{2011},False,"{vse10171, vse10191, vse10141, vse10181, typuz...",False,,True,Výběr údajů ze SLDB 2011 za domácnosti,jádrová pro IKŽ,více úrovní (ČR–kraj–okres–obec),False
116,117,B,Naděje dožití v okresech a správních obvodech ORP,130140r17,Naděje dožití v okresech a správních obvodech ...,LKOD,resource#1,{'cs': 'Data za rok 2017'},ds_130140r17.csv,NaN,...,"{2017, 2013}",False,"{vek_kod, vek_txt, casref_od, casref_do, vuzem...",True,vuzemi_kod,True,Naděje dožití v okresech a správních obvodech ...,jádrová pro IKŽ,ORP,False
123,124,C,Bilance meziokresní vyjížďky do zaměstnání pod...,170242,Bilance meziokresní vyjížďky do zaměstnání pod...,LKOD,resource#1,{'cs': 'Bilance meziokresní vyjížďky do zaměst...,ds_170242.csv,NaN,...,{2011},False,"{uzemido_kod, uzemido_cis, uzemiz_cis, stapro_...",False,,True,Bilance meziokresní vyjížďky do zaměstnání pod...,podpůrná / kontextová,okres,False
133,134,C,Průměrné spotřebitelské ceny vybraných výrobků...,12052,Průměrné spotřebitelské ceny vybraných výrobků...,DATASTAT,NaN,NaN,NaN,NaN,...,"{2018, 2006}",False,{},False,None,False,Průměrné spotřebitelské ceny vybraných výrobků...,podpůrná / kontextová,kraj,False
